# Week 41, Day 4: Optimal feedback control

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w041/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Riccati gains and a reach

Write lqr_gains(A, B, Qf, R, T) using the backward Riccati recursion, returning the list of gains K_0 to K_(T-1) in forward order. Then simulate the reach from 10 cm short of the target with u = -K x.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2002)
dt, T, m = 0.01, 60, 1.0
A = np.array([[1, dt], [0, 1]]); B = np.array([[0], [dt / m]])
Qf = np.diag([1e4, 1e2]); R = np.array([[1e-4]])

def lqr_gains(A, B, Qf, R, T):
    S = Qf.copy(); K_list = []
    for t in range(T):
        K = ...
        S = ...
        K_list.append(K)
    return K_list[::-1]

gains = lqr_gains(A, B, Qf, R, T)
x = np.array([-0.1, 0.0]); path = [x.copy()]
for t in range(T):
    x = A @ x + B @ (-gains[t] @ x); path.append(x.copy())
path = np.array(path)
print(f"final error {1000 * path[-1, 0]:.2f} mm, final velocity {path[-1, 1]:.4f} m/s; position gain from {gains[0][0, 0]:.1f} at the start to {max(g[0, 0] for g in gains):.0f} near the end")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(np.arange(T + 1) * dt, path[:, 1]); ax[0].set_title("velocity (m/s)")
ax[1].plot(np.arange(T) * dt, [g[0, 0] for g in gains]); ax[1].set_title("position gain")
for a in ax: a.set_xlabel("s")
plt.show()

In [ ]:
import numpy as np
def test_reach():
    "the reach ends within 1 mm of the target and nearly at rest"
    assert abs(path[-1, 0]) < 1e-3 and abs(path[-1, 1]) < 0.01

def test_gains():
    "the position gain rises steeply towards the end, peaking in the last few steps"
    k = [g[0, 0] for g in gains]
    assert np.argmax(k) >= T - 5 and max(k) > 20 * k[0]

test_reach()
test_gains()
print("All checks passed")

### Exercise: Correcting a push

Simulate the reach twice with a push of +0.3 m/s added to the velocity at step 30: once with feedback (u = -K x at every step) and once replaying the commands u_planned recorded from the unperturbed feedback run. Store both paths.

In [ ]:
u_planned = [(-gains[t] @ p)[0] for t, p in enumerate(path[:-1])]

def simulate(feedback):
    x = np.array([-0.1, 0.0]); out = [x.copy()]
    for t in range(T):
        u = ...
        x = A @ x + B @ np.array([u])
        if t == 30:
            x = x + np.array([0.0, 0.3])
        out.append(x.copy())
    return np.array(out)

fb, ol = simulate(True), simulate(False)
print(f"final error after the push: feedback {1000 * fb[-1, 0]:.2f} mm, open-loop replay {1000 * ol[-1, 0]:.1f} mm")
plt.plot(np.arange(T + 1) * dt, path[:, 0], "k--", label="unperturbed"); plt.plot(np.arange(T + 1) * dt, fb[:, 0], label="feedback"); plt.plot(np.arange(T + 1) * dt, ol[:, 0], label="open loop")
plt.axhline(0, color="gray", lw=0.5); plt.xlabel("s"); plt.ylabel("position relative to target (m)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_feedback():
    "with feedback, the perturbed reach still ends within 2 mm of the target"
    assert abs(fb[-1, 0]) < 2e-3

def test_openloop():
    "replaying the planned commands leaves an error larger than 5 mm"
    assert abs(ol[-1, 0]) > 5e-3

test_feedback()
test_openloop()
print("All checks passed")

### Exercise: Variability where it does not matter

Write the terminal cost matrices for the augmented state [x1, x2, 1]: Qf_opt = 1000 v vᵀ with v = [1, 1, -1], and Qf_servo = 1000 (v1 v1ᵀ + v2 v2ᵀ) with v1 = [1, 0, -0.5] and v2 = [0, 1, -0.5]. The simulation is given.

In [ ]:
A2 = np.eye(3); B2 = np.array([[dt, 0], [0, dt], [0, 0]]); R2 = 1e-3 * np.eye(2)
v = np.array([1, 1, -1.0]); v1 = np.array([1, 0, -0.5]); v2 = np.array([0, 1, -0.5])
Qf_opt = ...
Qf_servo = ...

def run(Qf, trials=300):
    Ks = lqr_gains(A2, B2, Qf, R2, T); ends, effort = [], []
    for _ in range(trials):
        x = np.array([0, 0, 1.0]); e = 0.0
        for t in range(T):
            u = -Ks[t] @ x; e += np.sum(u ** 2) * dt
            x = A2 @ x + B2 @ u + np.r_[0.02 * r.normal(size=2), 0]
        ends.append(x[:2]); effort.append(e)
    ends = np.array(ends)
    return {"relevant": ends.sum(1).var(), "irrelevant": (ends[:, 0] - ends[:, 1]).var(), "effort": np.mean(effort), "ends": ends}

res = {"optimal": run(Qf_opt), "servo": run(Qf_servo)}
for name, rr in res.items():
    print(f"{name:7s}: variance along the task {rr['relevant']:.5f}, across it {rr['irrelevant']:.5f}; effort {rr['effort']:.2f}")
for name, rr in res.items():
    plt.plot(rr["ends"][:, 0], rr["ends"][:, 1], ".", alpha=0.5, label=name)
plt.plot([0.2, 0.8], [0.8, 0.2], "k--", lw=0.8); plt.axis("equal"); plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_structure():
    "the optimal controller's irrelevant variance is more than 10 times its relevant variance; the servo's is within a factor of 2"
    assert res["optimal"]["irrelevant"] > 10 * res["optimal"]["relevant"]
    assert 0.5 < res["servo"]["irrelevant"] / res["servo"]["relevant"] < 2

def test_same_accuracy():
    "both controllers achieve similar task accuracy, and the optimal one uses less effort"
    assert res["optimal"]["relevant"] < 1.5 * res["servo"]["relevant"] and res["optimal"]["effort"] < res["servo"]["effort"]

test_structure()
test_same_accuracy()
print("All checks passed")